In [ ]:
import unittest
import os
import sys
import numpy as np

try:
    import segyio
except ImportError:
    # We still need segyio for the constants/enums even if we are testing npy
    raise ImportError("Error: segyio library is missing. Please install it.")

class NumpySegyAdapter:
    """
    Adapts a NumPy array (.npy) to look and behave like a segyio file handle.
    Allows running trace data integrity tests on raw numpy dumps.
    """
    def __init__(self, filepath):
        self.filepath = filepath
        # Load mmap_mode='r' to handle huge files without loading all into RAM
        try:
            self._data = np.load(filepath, mmap_mode='r')
        except Exception as e:
            raise RuntimeError(f"Could not load numpy file: {e}")

        # Handle Dimensions
        self.ndim = self._data.ndim
        if self.ndim == 2:
            # shape: (n_traces, n_samples)
            self.tracecount = self._data.shape[0]
            self.n_samples = self._data.shape[1]
            self._shape_3d = None
        elif self.ndim == 3:
            # shape: (n_il, n_xl, n_samples) -> Standard Seismic Cube
            self.n_il, self.n_xl, self.n_samples = self._data.shape
            self.tracecount = self.n_il * self.n_xl
            self._shape_3d = (self.n_il, self.n_xl)
        else:
            raise ValueError(f"Numpy array must be 2D or 3D, got {self.ndim}D")

        self.samples = np.arange(self.n_samples)
        
        # Mock Geometry Properties
        if self._shape_3d:
            self.ilines = np.arange(self.n_il)
            self.xlines = np.arange(self.n_xl)
        else:
            self.ilines = np.array([1])
            self.xlines = np.array([1])

        # Mock Headers
        self.text = [b"MOCK EBCDIC FOR NUMPY ARRAY"]
        # Standard SEGY uses microseconds (4000 = 4ms)
        self.bin = {
            segyio.BinField.Format: 5, # IEEE Float
            segyio.BinField.Interval: 4000, 
            segyio.BinField.Samples: self.n_samples
        }

    @property
    def trace(self):
        return self._TraceAccessor(self)

    class _TraceAccessor:
        def __init__(self, parent): self.parent = parent
        def __getitem__(self, idx):
            if self.parent.ndim == 2:
                return self.parent._data[idx]
            # Map flat index to 3D (il, xl)
            # Assuming row-major flattening (C-style)
            n_xl = self.parent.n_xl
            il = idx // n_xl
            xl = idx % n_xl
            return self.parent._data[il, xl]

    def attributes(self, field):
        """Generates synthetic attributes to satisfy test requirements."""
        if field == segyio.TraceField.INLINE_3D and self._shape_3d:
            # Repeat IL for every XL: [0,0,0, 1,1,1...]
            return np.repeat(np.arange(self.parent.n_il), self.parent.n_xl)
        
        if field == segyio.TraceField.CROSSLINE_3D and self._shape_3d:
            # Tile XL for every IL: [0,1,2, 0,1,2...]
            return np.tile(np.arange(self.parent.n_xl), self.parent.n_il)
            
        if field == segyio.TraceField.TraceIdentificationCode:
            return np.ones(self.tracecount, dtype=np.int32)
            
        if field == segyio.TraceField.SourceGroupScalar:
            return np.ones(self.tracecount, dtype=np.int32) # Return 1 (no scalar)

        # Default: Return zeros for coordinates etc. 
        # Tests relying on these should check type before asserting non-zero.
        return np.zeros(self.tracecount, dtype=np.int32)

    def __enter__(self): return self
    def __exit__(self, *args): pass


class SEGYIntegrityTests(unittest.TestCase):
    # Paths for Source (Original) and Target (Transferred) files
    # Supports .sgy, .segy, and .npy
    source_file = "sliepner.npy" 
    target_file = "sliepner.npy"
    
    # Optional ranges for strict validation
    expected_il_range = None
    expected_xl_range = None
    
    def setUp(self):
        """Checks if both source and target files exist before running tests."""
        if not os.path.exists(self.source_file):
            self.skipTest(f"Source file {self.source_file} not found.")
        if not os.path.exists(self.target_file):
            self.skipTest(f"Target file {self.target_file} not found.")

    def safe_open_segy(self, filepath, **kwargs):
        """
        Helper to open SEGY or NumPy files safely. 
        Returns a segyio handle or a NumpySegyAdapter.
        """
        # 1. Handle NumPy Files
        if filepath.endswith('.npy'):
            try:
                return NumpySegyAdapter(filepath)
            except Exception as e:
                self.fail(f"Failed to open NumPy file '{filepath}': {e}")

        # 2. Handle SEGY Files
        try:
            return segyio.open(filepath, **kwargs)
        except RuntimeError as e:
            # Diagnostics for SEGY failures
            msg = str(e)
            if "unable to count traces" in msg or "no data traces" in msg:
                file_size = os.stat(filepath).st_size
                with open(filepath, 'rb') as f:
                    sig = f.read(6)
                
                if sig == b'\x93NUMPY':
                    # If user forgot .npy extension but it IS numpy
                    return NumpySegyAdapter(filepath)
                else:
                    self.fail(f"SEGY Structure Error: {msg}.\n"
                              f"File: {filepath} ({file_size} bytes).")
            raise e

    def is_numpy(self, handle):
        """Helper to detect if we are testing a numpy adapter."""
        return isinstance(handle, NumpySegyAdapter)

    def test_00_format_check(self):
        """Info check on file formats."""
        for fpath in [self.source_file, self.target_file]:
            if fpath.endswith('.npy'):
                print(f"[INFO] '{fpath}' identified as NumPy. Using Adapter.")

    def test_01_ebcdic_header_readable(self):
        with self.safe_open_segy(self.target_file, ignore_geometry=True) as f:
            if self.is_numpy(f):
                # Numpy doesn't have a real EBCDIC header, so we skip strictly checking content
                # but we check if our adapter provides the mock property correctly.
                self.assertTrue(hasattr(f, 'text'))
            else:
                text = f.text[0]
                self.assertIsInstance(text, (str, bytes))
                self.assertEqual(len(text), 3200)

    def test_02_binary_header_sanity(self):
        with self.safe_open_segy(self.target_file, ignore_geometry=True) as f:
            # Handle Interval (segyio tool helper vs dict access)
            if self.is_numpy(f):
                sample_interval = f.bin[segyio.BinField.Interval] / 1000.0
            else:
                sample_interval = segyio.tools.dt(f) / 1000.0
                
            ns = f.samples.size
            self.assertTrue(sample_interval > 0, "Sample interval is zero or negative")
            self.assertTrue(ns > 0, "Number of samples is zero")
            fmt = f.bin[segyio.BinField.Format]
            self.assertIn(fmt, [1, 2, 3, 4, 5, 8], f"Unknown Format Code: {fmt}")

    def test_03_filesize_consistency(self):
        file_stats = os.stat(self.target_file)
        actual_size = file_stats.st_size
        
        with self.safe_open_segy(self.target_file, ignore_geometry=True) as f:
            if self.is_numpy(f):
                # NPY size = header + data.
                # Data size = traces * samples * itemsize (usually 4 for float32)
                # We can't easily predict NPY header size, but data size should be slightly less than file size
                min_data_size = f.tracecount * f.n_samples * 4 
                self.assertTrue(actual_size >= min_data_size, 
                                f"Numpy file too small for declared shape. Size: {actual_size}, Min Data: {min_data_size}")
            else:
                trace_count = f.tracecount
                sample_count = f.samples.size
                fmt = f.bin[segyio.BinField.Format]
                bytes_per_sample = 1 if fmt == 8 else 4
                expected_size = 3600 + trace_count * (240 + (sample_count * bytes_per_sample))
                diff = abs(actual_size - expected_size)
                self.assertTrue(diff < 4096, f"Size mismatch. Expected ~{expected_size}, got {actual_size}")

    def test_04_3d_geometry_strict(self):
        with self.safe_open_segy(self.target_file, "r", strict=True) as f:
            ilines = f.ilines
            xlines = f.xlines
            self.assertTrue(len(ilines) > 0)
            self.assertTrue(len(xlines) > 0)
            if self.expected_il_range:
                self.assertTrue(min(ilines) >= self.expected_il_range[0])
                self.assertTrue(max(ilines) <= self.expected_il_range[1])

    def test_05_data_integrity_sampling(self):
        with self.safe_open_segy(self.target_file, ignore_geometry=True) as f:
            trace_count = f.tracecount
            indices = np.random.randint(0, trace_count, min(10, trace_count))
            for idx in indices:
                trace_data = f.trace[idx]
                if np.isnan(trace_data).any():
                    self.fail(f"NaN found in Target trace {idx}")
                if np.isinf(trace_data).any():
                    self.fail(f"Inf found in Target trace {idx}")

    def test_06_unique_grid_locations(self):
        with self.safe_open_segy(self.target_file, ignore_geometry=True) as f:
            # Adapter generates synthetic grid for 3D arrays, so this validates dimensions
            ils = f.attributes(segyio.TraceField.INLINE_3D)[:]
            xls = f.attributes(segyio.TraceField.CROSSLINE_3D)[:]
            pairs = set(zip(ils, xls))
            self.assertEqual(len(pairs), f.tracecount, "Duplicate Inline/Crossline pairs found in Target")

    def test_07_coordinate_scalars(self):
        with self.safe_open_segy(self.target_file, ignore_geometry=True) as f:
            if self.is_numpy(f): return # Skip for NumPy
            
            scalars = f.attributes(segyio.TraceField.SourceGroupScalar)[:]
            non_zero = np.count_nonzero(scalars)
            self.assertTrue(non_zero > 0.9 * len(scalars), "Excessive zero coordinate scalars in Target")

    def test_08_coordinate_validity(self):
        with self.safe_open_segy(self.target_file, ignore_geometry=True) as f:
            if self.is_numpy(f): return # Skip for NumPy (coordinates don't exist)
            
            sx = f.attributes(segyio.TraceField.SourceX)[:]
            sy = f.attributes(segyio.TraceField.SourceY)[:]
            self.assertFalse(np.all(sx == 0), "All SourceX are zero in Target")
            self.assertFalse(np.all(sy == 0), "All SourceY are zero in Target")

    def test_09_trace_id_code(self):
        with self.safe_open_segy(self.target_file, ignore_geometry=True) as f:
            tids = f.attributes(segyio.TraceField.TraceIdentificationCode)[:]
            valid_ids = np.isin(tids, [1, 2])
            self.assertTrue(np.all(valid_ids), "Invalid Trace ID codes found in Target")

    def test_10_amplitude_spikes(self):
        with self.safe_open_segy(self.target_file, ignore_geometry=True) as f:
            indices = np.random.randint(0, f.tracecount, min(50, f.tracecount))
            for idx in indices:
                tr = f.trace[idx]
                if np.max(np.abs(tr)) > 1e30:
                     self.fail(f"Amplitude spike > 1e30 at Target trace {idx}")

    def test_11_inline_step_consistency(self):
        """Checks if Inlines increase by a constant step size."""
        with self.safe_open_segy(self.target_file, ignore_geometry=True) as f:
            ils = np.sort(np.unique(f.attributes(segyio.TraceField.INLINE_3D)[:]))
            if len(ils) > 1:
                steps = np.diff(ils)
                unique_steps = np.unique(steps)
                self.assertEqual(len(unique_steps), 1, f"Irregular Inline spacing detected in Target: {unique_steps}")

    def test_12_crossline_step_consistency(self):
        """Checks if Crosslines increase by a constant step size."""
        with self.safe_open_segy(self.target_file, ignore_geometry=True) as f:
            xls = np.sort(np.unique(f.attributes(segyio.TraceField.CROSSLINE_3D)[:]))
            if len(xls) > 1:
                steps = np.diff(xls)
                unique_steps = np.unique(steps)
                self.assertEqual(len(unique_steps), 1, f"Irregular Crossline spacing detected in Target: {unique_steps}")

    def test_13_dead_trace_statistics(self):
        """Fails if > 50% of sampled traces are dead (all zeros)."""
        sample_size = 100
        dead_count = 0
        with self.safe_open_segy(self.target_file, ignore_geometry=True) as f:
            count = min(sample_size, f.tracecount)
            indices = np.linspace(0, f.tracecount - 1, count, dtype=int)
            
            for idx in indices:
                if np.all(f.trace[idx] == 0):
                    dead_count += 1
            
            percentage = (dead_count / count) * 100
            self.assertTrue(percentage < 50.0, f"Dead trace percentage in Target too high: {percentage:.1f}%")

    def test_14_affine_geometry_check(self):
        """
        Checks if X/Y coordinates map linearly to Inline/Crossline.
        """
        with self.safe_open_segy(self.target_file, ignore_geometry=True) as f:
            if self.is_numpy(f): return # Skip for NumPy (No real coords)
            
            # Sample 500 random points to speed up calculation
            count = min(500, f.tracecount)
            indices = np.linspace(0, f.tracecount - 1, count, dtype=int)
            
            il = f.attributes(segyio.TraceField.INLINE_3D)[indices]
            xl = f.attributes(segyio.TraceField.CROSSLINE_3D)[indices]
            sx = f.attributes(segyio.TraceField.SourceX)[indices]
            sy = f.attributes(segyio.TraceField.SourceY)[indices]
            
            A = np.vstack([il, xl, np.ones(len(il))]).T
            try:
                _, resid_x, _, _ = np.linalg.lstsq(A, sx, rcond=None)
                _, resid_y, _, _ = np.linalg.lstsq(A, sy, rcond=None)
                mse_x = resid_x[0] / count if len(resid_x) > 0 else 0
                mse_y = resid_y[0] / count if len(resid_y) > 0 else 0
                
                self.assertTrue(mse_x < 1000, f"Non-Affine Geometry detected (X-resid: {mse_x})")
                self.assertTrue(mse_y < 1000, f"Non-Affine Geometry detected (Y-resid: {mse_y})")
            except Exception as e:
                print(f"  [Warning] Affine check skipped: {e}")

    def test_15_compare_source_target_metadata(self):
        """
        DIRECT COMPARISON: Verifies Source matches Target.
        """
        # 1. File Size check (Relaxed for NPY vs SGY comparison)
        s_size = os.stat(self.source_file).st_size
        t_size = os.stat(self.target_file).st_size
        
        # 2. Structure check
        with self.safe_open_segy(self.source_file, ignore_geometry=True) as s, \
             self.safe_open_segy(self.target_file, ignore_geometry=True) as t:
            
            self.assertEqual(s.tracecount, t.tracecount, 
                             f"Trace count mismatch! Source: {s.tracecount}, Target: {t.tracecount}")
            self.assertEqual(s.samples.size, t.samples.size, 
                             f"Sample count mismatch! Source: {s.samples.size}, Target: {t.samples.size}")

if __name__ == '__main__':
    print(f"Starting SEGY/Numpy Integrity Tests.")
    print(f"Source: {SEGYIntegrityTests.source_file}")
    print(f"Target: {SEGYIntegrityTests.target_file}")
    
    suite = unittest.TestSuite()
    for method in dir(SEGYIntegrityTests):
        if method.startswith("test_"):
            suite.addTest(SEGYIntegrityTests(method))
    
    runner = unittest.TextTestRunner(verbosity=2)
    result = runner.run(suite)
    
    if not result.wasSuccessful():
        print("\n[FAILURE] Some tests failed. Please review the output above.")
    else:
        print("\n[SUCCESS] All tests passed.")